#  Retrieval-Augmented Generation (RAG) Chatbot

This project implements a Retrieval-Augmented Generation (RAG) system that enables users to ask questions based on a PDF document. The system retrieves relevant content using semantic search and generates answers using a local language model.


## Problem Statement

Traditional document search methods rely on keyword matching, which often fails to capture the semantic meaning of user queries. This results in irrelevant or incomplete answers.

There is a need for an intelligent system that:
- Understands user queries contextually
- Retrieves relevant information from documents
- Generates human-like answers

This project solves this by combining semantic search with language generation using a RAG architecture.


##  Objectives

- To build a system that can answer questions from PDF documents
- To implement semantic search using embeddings
- To use FAISS for efficient vector storage and retrieval
- To integrate a local LLM for answer generation
- To avoid dependency on paid APIs (fully local solution)

##  Requirements

### Libraries Used:
- LangChain
- FAISS
- HuggingFace Transformers
- Sentence Transformers
- PyPDF Loader

### System Requirements:
- Python 3.9+

##  Project Workflow

1. Load PDF document
2. Split text into smaller chunks
3. Convert text into embeddings using MiniLM
4. Store embeddings in FAISS vector database
5. Retrieve relevant chunks based on user query
6. Pass retrieved context to LLM
7. Generate final answer

This pipeline follows the RAG (Retrieval-Augmented Generation) architecture.

In [1]:
!pip install langchain openai faiss-cpu pypdf

^C


## Data Loading

In this step, we load the PDF document using PyPDFLoader.  
Each page of the PDF is treated as a document for further processing.

In [3]:
!pip install langchain_community

  Using cached langchain-1.2.12-py3-none-any.whl.metadata (5.7 kB)
  Using cached faiss_cpu-1.13.2-cp313-cp313-win_amd64.whl.metadata (7.6 kB)
  Using cached pypdf-6.9.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-1.2.19-py3-none-any.whl.metadata (4.4 kB)
  Using cached langgraph-1.1.2-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl.metadata (5.2 kB)
  Using cach


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


^C


In [4]:
from langchain_community.document_loaders import PyPDFLoader

In [5]:


loader = PyPDFLoader("data/Data Science_Machine learning.pdf")
documents = loader.load()

print("Number of pages:", len(documents))
print(documents[0].page_content[:300])

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_classic-1.0.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached sqlalchemy-2.0.48-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached aiohttp-3.13.3-cp313-cp313-win_amd64.whl.metadata (8.4 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached frozenlist-1.8.0-cp313-cp313-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.7.1-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached propcache-0.4.1-cp313-cp313-win_amd64.whl.metadata (14 kB)
  Using cached yarl-1.23.0-cp313-cp313-win_amd64.whl.metadata


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Number of pages: 245



### Text Splitting

Large documents are split into smaller chunks to:
- Improve retrieval accuracy
- Reduce memory usage
- Enable better embedding representation

We use RecursiveCharacterTextSplitter for efficient chunking.

In [ ]:
!pip install langchain-text-splitters


In [7]:
from langchain_text_splitters import CharacterTextSplitter

In [10]:


text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

texts = text_splitter.split_documents(documents)

print("Number of chunks:", len(texts))
print(texts[0].page_content[:300])

Number of chunks: 239
PYTHON MACHINE LEARNING: A
BEGINNER'S GUIDE TO SCIKIT-
LEARN


## Embeddings

Text chunks are converted into numerical vectors using HuggingFace embeddings (MiniLM).

These embeddings capture semantic meaning, allowing similarity-based search instead of keyword matching.

In [13]:
!pip install sentence-transformers

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached regex-2026.2.28-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markup

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\USER\\OneDrive\\Desktop\\RAG\\rag-knowledge-assistant\\venv\\Lib\\site-packages\\networkx\\algorithms\\centrality\\flow_matrix.py'
Check the permissions.



In [14]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [17]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Embeddings loaded successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1423.27it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings loaded successfully


## Vector Database (FAISS)

FAISS is used to store and index embeddings for fast similarity search.

It allows efficient retrieval of relevant text chunks based on user queries.

In [23]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)
print("Vector store created successfully")

Vector store created successfully


In [24]:
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 3})

print("Retriever ready")

Retriever ready


In [39]:
!pip install transformers accelerate


In [40]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

In [41]:
pipe = pipeline("text-generation",
                model="google/flan-t5-base",   # lightweight & good
                max_length=512,
                temperature=0)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM loaded (FREE)")

c:\Users\USER\OneDrive\Desktop\RAG\rag-knowledge-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 282/282 [00:00<00:00, 913.22it/s]
The tied weights m

LLM loaded (FREE)


C:\Users\USER\AppData\Local\Temp\ipykernel_23416\1507553332.py:6: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [44]:
!pip install -U langchain

In [47]:
!pip install langchain==0.1.20
!pip install langchain-community==0.0.38
!pip install langchain-core==0.1.52

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\USER\OneDrive\Desktop\RAG\rag-knowledge-assistant\venv\Scripts\python.exe C:\Users\USER\AppData\Local\Temp\pip-install-u6a3c45v\numpy_20a95d77b7914fca9edf577f9a72fabc\vendored-meson\meson\meson.py setup C:\Users\USER\AppData\Local\Temp\pip-install-u6a3c45v\numpy_20a95d77b7914fca9edf577f9a72fabc C:\Users\USER\AppData\Local\Temp\pip-install-u6a3c45v\numpy_20a95d77b7914fca9edf577f9a72fabc\.mesonpy-cee1ec7l -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\USER\AppData\Local\Temp\pip-install-u6a3c45v\numpy_20a95d77b7914fca9edf577f9a72fabc\.mesonpy-cee1ec7l\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\USER\AppData\Local\Temp\pip-install-u6a3c45v\numpy_20a95d77b7914fca9edf577f9a72fabc
      Build dir: C:\Users\USER\AppData\Local\Temp\pip-

     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.8/15.8 MB 9.5 MB/s eta 0:00:02
     --------- ------------------------------ 3.7/15.8 MB 9.3 MB/s eta 0:00:02
     -------------- ------------------------- 5.8/15.8 MB 9.2 MB/s eta 0:00:02
     ------------------- -------------------- 7.6/15.8 MB 9.2 MB/s eta 0:00:01
     ------------------------ --------------- 9.7/15.8 MB 9.2 MB/s eta 0:00:01
     ----------------------------- ---------- 11.5/15.8 MB 9.2 MB/s eta 0:00:01
     --------------------------------- ------ 13.4/15.8 MB 9.2 MB/s eta 0:00:01
     ---------------------------------------  15.5/15.8 MB 9.2 MB/s eta 0:00:01
     ---------------------------------------- 15.8/15.8 MB 8.6 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + C:\Users\USER\OneDrive\Desktop\RAG\rag-knowledge-assistant\venv\Scripts\python.exe C:\Users\USER\AppData\Local\Temp\pip-install-twz5j9z8\numpy_ba180bbc9a0d4ad5b7ac6e7645f043df\vendored-meson\meson\meson.py setup C:\Users\USER\AppData\Local\Temp\pip-install-twz5j9z8\numpy_ba180bbc9a0d4ad5b7ac6e7645f043df C:\Users\USER\AppData\Local\Temp\pip-install-twz5j9z8\numpy_ba180bbc9a0d4ad5b7ac6e7645f043df\.mesonpy-ty7swzn6 -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\USER\AppData\Local\Temp\pip-install-twz5j9z8\numpy_ba180bbc9a0d4ad5b7ac6e7645f043df\.mesonpy-ty7swzn6\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\USER\AppData\Local\Temp\pip-install-twz5j9z8\numpy_ba180bbc9a0d4ad5b7ac6e7645f043df
      Build dir: C:\Users\USER\AppData\Local\Temp\pip-

  Using cached langchain_community-0.0.38-py3-none-any.whl.metadata (8.7 kB)
  Using cached langchain_core-0.1.53-py3-none-any.whl.metadata (5.9 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)

  Attempting uninstall: tenacity

    Found existing installation: tenacity 9.1.4

    Uninstalling tenacity-9.1.4:

      Successfully uninstalled tenacity-9.1.4

   ---------------------------------------- 0/4 [tenacity]
   --------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.3 requires langchain-core<2.0.0,>=1.2.19, but you have langchain-core 0.1.52 which is incompatible.
langchain-openai 1.1.11 requires langchain-core<2.0.0,>=1.2.18, but you have langchain-core 0.1.52 which is incompatible.
langchain-text-splitters 1.1.1 requires langchain-core<2.0.0,>=1.2.13, but you have langchain-core 0.1.52 which is incompatible.
langgraph-checkpoint 4.0.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.52 which is incompatible.
langgraph-prebuilt 1.0.8 requires langchain-core>=1.0.0, but you have langchain-core 0.1.52 which is incompatible.


## Language Model (LLM)

A local HuggingFace model (GPT-2) is used to generate answers.

Advantages:
- No API cost
- Fully offline capability
- Easy integration

In [37]:
import os

from langchain_openai import ChatOpenAI


In [57]:
pipe = pipeline("text-generation",
                model="google/flan-t5-base",   # lightweight & good
                max_length=512,
                temperature=0)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM loaded (FREE)")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 661.26it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxFor

LLM loaded (FREE)


In [61]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="gpt2",
    max_length=200,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM working now")

c:\Users\USER\OneDrive\Desktop\RAG\rag-knowledge-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2647.75it/s]
The following generation flags ar

LLM working now


## Retrieval

The retriever searches the FAISS database and returns the most relevant chunks for a given query.

This ensures that only useful context is passed to the language model.

In [66]:
query = "What is AI?"

docs = retriever.invoke(query)

context = " ".join([doc.page_content for doc in docs[:1]])

response = llm.invoke(f"""
You are a helpful assistant.
Answer briefly.
Context:
{context}
Question: {query}
Answer:""")

print(response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=150) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful assistant.
Answer briefly.
Context:
M
1.1  BACKGROUND ON MACHINE
LEARNING
achine learning is a subﬁeld of artiﬁcial intelligence (AI)
and is a powerful tool for solving complex problems in a
variety of industries, including ﬁnance, healthcare,
transportation, and more.
The history of machine learning can be traced back to the
1940s and 1950s, when researchers ﬁrst began exploring the
use of computers for problem-solving and decision-making. In
the early days of machine learning, algorithms were primarily
used for simple tasks, such as classiﬁcation and clustering.
However, as technology advanced and data became more
readily available, machine learning began to evolve and
expand into more complex applications.
One of the key milestones in the history of machine learning
was the development of the perceptron algorithm in the
1950s. The perceptron was the ﬁrst algorithm capable of
learning from data and was used for simple pattern
recognition tasks. This was followed by

##  Results

- Successfully retrieves relevant information from PDF
- Generates context-aware answers
- Works without external APIs
- Demonstrates real-world GenAI application

Limitations:
- GPT-2 produces basic responses
- Output quality depends on context size and model capability

##  Future Improvements

- Use advanced models (FLAN-T5 / OpenAI GPT)
- Add Streamlit UI for chatbot interface
- Support multiple PDFs
- Add chat history (memory)
- Improve response quality with better prompts

## Conclusion

This project demonstrates how Retrieval-Augmented Generation can be used to build intelligent document-based question-answering systems.

It highlights the integration of:
- NLP
- Vector databases
- Large Language Models

The system provides a strong foundation for real-world AI applications.